# Graph maintenance

A knowledge graph built from a corpus goes stale as the corpus changes: posts are withdrawn, answers are edited, a new model re-extracts old text, and the same thing appears under five spellings. This notebook maintains the ServerFault graph from `01_construction_provenance_store.ipynb` (63,436 entities and 86,644 relations from 9,999 posts) without rebuilding it, and measures what each operation changes:

1. Retraction by provenance: remove one post's facts and see what else moves.
2. Re-extraction of a revised post, using the real earlier revisions in the dump's `PostHistory.xml`.
3. Extraction drift: re-extract unchanged posts with a second model, and with the same model again.
4. Deduplication of near-duplicate entity names, with a merge you can measure.
5. A maintenance gate: schema checks, topology before and after, and smoke questions that must not cite a retracted post.
6. The library contrast: LightRAG's delete, merge, and edit run live on a small index, and HippoRAG 2, which has no update API.

The code is `aieng.advanced_rag.graphrag.maintenance`, and `graph_maintenance.py` runs each measurement and writes one JSON file to `outputs/maintenance/`. With `RUN_EXPENSIVE = False` the notebook reads those files. Every figure names the model that produced it. The outputs show entity names, counts, and document ids, and no post text, because ServerFault posts are CC BY-SA (see `CORPUS.md`).

In [1]:
import json
from pathlib import Path

import graph_maintenance as gm


# True re-runs every measurement. The model-free steps need the ServerFault graph,
# the slice, and the extraction response cache in the local artifact cache, and
# take a few minutes each. The model steps call the models named below through the
# proxy, at RPM requests a minute, and cache every response.
RUN_EXPENSIVE = False

# The model that built the graph. Revisions are re-extracted with it, so a changed
# fact reflects the edit and not a change of model.
REFERENCE_MODEL = gm.REFERENCE_MODEL
# Set this to your own model to measure how far it drifts from the graph.
DRIFT_MODEL = gm.DRIFT_MODEL
# The model that built the 5-passage LightRAG index.
LIGHTRAG_MODEL = gm.LIGHTRAG_MODEL
RPM = 15

OUT = Path.cwd() / "outputs" / "maintenance"
corpus = gm.Corpus() if RUN_EXPENSIVE else None


def load(name):
    """Read one committed result."""
    return json.loads((OUT / name).read_text())


print("reference model:", REFERENCE_MODEL)
print("drift model:    ", DRIFT_MODEL)
print("LightRAG model: ", LIGHTRAG_MODEL)

reference model: Qwen3_8-27B
drift model:     Qwen3_8-Flash-Next
LightRAG model:  Qwen3_6-35B-A3B


## Findings

The surprising result comes first. Re-extracting an unchanged post with the same model, the same prompt, and temperature 0 does not return the same graph. On 50 posts, `Qwen3_8-27B` run a second time keeps a mean 92% of its entities and 74% of its relations. That run-to-run churn is the floor any maintenance diff sits on, and the revision and drift results below read against it.

In [2]:
retraction, revision, drift = load("retraction.json"), load("revision.json"), load("drift.json")
dedup, labels, gate = load("dedup.json"), load("dedup_labels.json"), load("gate.json")
lightrag, hipporag = load("lightrag.json"), load("hipporag.json")

floor, other = drift["rerun_reference"], drift["second_model"]
print(
    f"same model again ({floor['model']}):  entity Jaccard {floor['entity_jaccard']['mean']:.2f}, relation Jaccard {floor['relation_jaccard']['mean']:.2f}"
)
print(
    f"second model ({other['model']}): entity Jaccard {other['entity_jaccard']['mean']:.2f}, relation Jaccard {other['relation_jaccard']['mean']:.2f}"
)
print(
    f"real revisions ({revision['model']}):     entity Jaccard {revision['entity_jaccard']['mean']:.2f}, relation Jaccard {revision['relation_jaccard']['mean']:.2f}"
)
s = retraction["sample"]["summary"]
print()
print(
    f"one retraction: {s['entities_removed']['mean']} entities removed, {s['neighbourhood']['mean']:.0f} neighbours whose context changes, 0 model calls"
)
print(
    f"one revision:   {revision['edited_window_calls'] / revision['documents']:.0f} call for the edited section, {revision['edited_window_tokens'] / revision['documents']:,.0f} tokens"
)
print("full rebuild:   20,077 calls, 26.1 million tokens, 8.1 hours")

same model again (Qwen3_8-27B):  entity Jaccard 0.92, relation Jaccard 0.74
second model (Qwen3_8-Flash-Next): entity Jaccard 0.61, relation Jaccard 0.19
real revisions (Qwen3_8-27B):     entity Jaccard 0.81, relation Jaccard 0.61

one retraction: 5.38 entities removed, 507 neighbours whose context changes, 0 model calls
one revision:   1 call for the edited section, 1,321 tokens
full rebuild:   20,077 calls, 26.1 million tokens, 8.1 hours


## A post in a chunk index and in a graph

Deleting a post from a dense chunk index removes its chunks, which are keyed by the post, and touches nothing else. In the graph, the same post contributed to entities that other posts also mention, so removing it changes shared nodes. The cell compares the two for the five posts this notebook retracts.

In [3]:
print(
    f"{'post':<9} {'chunks':>6} {'entities removed':>17} {'entities touched':>17} {'relations removed':>18} {'neighbours':>11}"
)
for doc_id, row in retraction["demo"].items():
    print(
        f"{doc_id:<9} {row['blocks']:>6} {row['entities_removed']:>17} {row['entities_touched']:>17} "
        f"{row['relations_removed']:>18} {row['neighbourhood']:>11}"
    )

post      chunks  entities removed  entities touched  relations removed  neighbours
q9708         21                 5                10                 12         849
q62411        18                 7                 8                 10         114
q309357        9                 1                 7                  4         710
q264595        8                 4                 4                  5         776
q253313       22                 7                 9                  8         947


## Retraction by provenance

Every mention in the graph names the post it came from, so a retraction removes the post's mentions, drops each entity and relation left with none, and drops any relation whose endpoint went with it. A surviving entity can still carry a description or a type that the retracted post wrote, because `KnowledgeGraph` keeps the longest description and the first type without saying where they came from. `extract_units` rebuilds each post's own extraction from the response cache at no cost, and `retract(..., units=units)` then recomputes the description and type from the posts that remain.

Post `q9708` explains the PEM file format. Retracting it removes the five formats only it mentions and touches ten entities that other posts share, including `openssl` (169 relations) and `windows` (499).

In [4]:
if RUN_EXPENSIVE:
    gm.write("retraction.json", gm.step_retraction(corpus))
    retraction = load("retraction.json")

row = retraction["demo"]["q9708"]
print("removed:", ", ".join(row["entities_removed_names"]))
print("touched hubs (relations):", ", ".join(f"{n} ({d})" for n, d in row["hub_degrees_touched"]))
print(f"descriptions repaired from the remaining posts: {row['descriptions_repaired']}")
print(f"types that came from q9708 and changed:         {row['types_changed']}")
print(f"facts still citing q9708 afterwards:            {row['remaining_citations']}")
print()
fold = retraction["fold_of_units_against_build"]
print(
    f"retracting all five from the folded graph equals folding the other 9,994 posts: {retraction['retract_equals_rebuild_of_remaining_units']}"
)
print(
    f"relations the single build pass kept that a per-post fold does not: {fold['relations_removed']} of {retraction['graph']['relations']:,}"
)

removed: .crl, .csr, .der, .pkcs12, pkcs10
touched hubs (relations): windows (499), debian (219), openssl (169), java (57), .pem (7)
descriptions repaired from the remaining posts: 4
types that came from q9708 and changed:         4
facts still citing q9708 afterwards:            0

retracting all five from the folded graph equals folding the other 9,994 posts: True
relations the single build pass kept that a per-post fold does not: 151 of 86,644


The last line is an order dependence in the original build. A single `build_graph` pass accepts a relation when both endpoints exist anywhere in the graph so far, so 151 relations in one post point at an entity that only an earlier post extracted. Folding per-post units drops them, and a retraction of that earlier post leaves them dangling, which `retract` removes and counts.

### Retraction at scale

The same measurement for 500 posts drawn with seed 0, each retracted alone from the full graph. A stale description is one that only the retracted post wrote, which a retraction without units would leave in place.

In [5]:
summary = retraction["sample"]["summary"]
print(f"{'per retracted post':<40} {'mean':>7} {'median':>7} {'max':>6}")
names = {
    "entities_removed": "entities removed",
    "entities_touched": "entities that lose some support",
    "relations_removed": "relations removed",
    "relations_dangling": "relations dropped, endpoint gone",
    "relations_touched": "relations that lose some support",
    "neighbourhood": "one-hop neighbours of changed entities",
    "stale_descriptions": "descriptions written only by the post",
}
for key, label in names.items():
    v = summary[key]
    print(f"{label:<40} {v['mean']:>7} {v['median']:>7} {v['max']:>6}")
print()
print(
    f"{retraction['single_document_entities']:,} of {retraction['graph']['entities']:,} entities are supported by one post"
)
print("largest hubs:", ", ".join(f"{n} ({d})" for n, d in retraction["top_hubs"][:5]))

per retracted post                          mean  median    max
entities removed                            5.38     5.0     29
entities that lose some support             7.57     7.0     22
relations removed                           8.61     8.0     25
relations dropped, endpoint gone             0.0     0.0      1
relations that lose some support             0.7     0.0      7
one-hop neighbours of changed entities    507.49   407.5   2037
descriptions written only by the post       1.34     1.0      9

51,971 of 63,436 entities are supported by one post
largest hubs: nginx (1018), ssh (990), linux (730), apache (686), windows (499)


A retraction removes about five entities and changes the one-hop context of about 500 others, because posts touch hubs such as `nginx` and `ssh`. For a retriever that expands one hop from a linked entity, those 500 neighbours are the blast radius of one withdrawn post.

## Re-extraction of a revised post

The pinned dump ships `PostHistory.xml`, which holds every revision of every post as Markdown. `graph_maintenance.py` reads the history of the slice's 19,999 questions and accepted answers, renders each revision with the same HTML stripping the corpus loader uses, and keeps posts whose latest revision renders to exactly the text the graph was built from. The earlier version of a candidate then differs from the graph's document by the edit and nothing else. 25 documents were drawn with seed 0, and each one's earlier version was extracted with the reference model, which is the graph as it stood before the edit.

The unchanged section of a revised post has the same prompt as before, so only the edited section costs a call.

In [6]:
if RUN_EXPENSIVE:
    gm.write("revision.json", gm.step_revision(corpus, RPM, model=REFERENCE_MODEL))
    revision = load("revision.json")

print(f"model {revision['model']}")
print(
    f"{revision['candidates']['posts']:,} edited posts qualify, in {revision['candidates']['documents']:,} documents; {revision['documents']} drawn"
)
print(
    f"{revision['edited_window_calls']} calls for edited sections, {revision['edited_window_tokens']:,} tokens; {revision['unchanged_windows_reused']} unchanged sections reused from the cache"
)
t = revision["totals"]
print()
print(
    f"{'':<12} {'added':>6} {'at the edit':>12} {'elsewhere':>10} {'unlocated':>10} {'removed':>8} {'at the edit':>12} {'elsewhere':>10} {'unlocated':>10}"
)
for kind in ("entities", "relations"):
    print(
        f"{kind:<12} {t[f'{kind}_added']:>6} {t[f'added_{kind}_at_edit']:>12} {t[f'added_{kind}_elsewhere']:>10} {t[f'added_{kind}_unlocated']:>10} "
        f"{t[f'{kind}_removed']:>8} {t[f'removed_{kind}_at_edit']:>12} {t[f'removed_{kind}_elsewhere']:>10} {t[f'removed_{kind}_unlocated']:>10}"
    )
print()
print(
    f"after applying all 25 revisions: {revision['after_revisions_validation_problems']} validation problems, entity Jaccard against the current graph {revision['after_revisions_against_current_fold']['entity_jaccard']}"
)

model Qwen3_8-27B
3,393 edited posts qualify, in 3,053 documents; 25 drawn
25 calls for edited sections, 33,020 tokens; 25 unchanged sections reused from the cache

              added  at the edit  elsewhere  unlocated  removed  at the edit  elsewhere  unlocated
entities         47           39          8          0       30           19         11          0
relations        68           31         32          5       52           17         26          9

after applying all 25 revisions: 0 validation problems, entity Jaccard against the current graph 1.0


A fact is at the edit when one of its quotes sits within 40 characters of a word the edit changed. 39 of 47 added entities sit at the edit, so entity changes mostly follow the text. Relations do not: about half of the added and removed relations sit away from the edit. The noise floor in the next section accounts for that, since the same model on identical text already changes about a quarter of its relations.

Applying the 25 revisions to the earlier graph with `revise` returns exactly the current graph's entities and relations, with no validation problems. Seven entities end with a different type, because `revise` folds a new version in after every other post and an entity keeps the type of the first post that named it.

## Extraction drift

Participants extract with their own model while the cached graph came from `Qwen3_8-27B`. Mixing the two in one graph is a maintenance problem: a post re-extracted by a second model changes even where its text did not. 50 unchanged posts (seed 1) were re-extracted twice: once by the drift model and once by the reference model again, with a request tag that keeps the second run out of the original cache entries.

In [7]:
if RUN_EXPENSIVE:
    gm.write("drift.json", gm.step_drift(corpus, RPM, drift_model=DRIFT_MODEL, reference=REFERENCE_MODEL))
    drift = load("drift.json")

print(
    f"{'re-extraction of 50 unchanged posts':<38} {'entity J':>9} {'median':>7} {'relation J':>11} {'median':>7} {'type changes':>13} {'calls':>6} {'tokens':>8}"
)
for label in ("rerun_reference", "second_model"):
    r = drift[label]
    print(
        f"{r['model']:<38} {r['entity_jaccard']['mean']:>9.2f} {r['entity_jaccard']['median']:>7.2f} "
        f"{r['relation_jaccard']['mean']:>11.2f} {r['relation_jaccard']['median']:>7.2f} "
        f"{r['type_changes_on_shared_entities']:>6} of {r['shared_entities']:<4} {r['cold_calls']:>6} {r['cold_tokens']:>8,}"
    )
print()
print(
    f"entities per post: reference {drift['reference_entities_per_post']['mean']}, drift model {drift['second_model']['entities_per_post']['mean']}"
)
w = drift["second_model"]["whole_graph_if_swapped"]
print(
    f"swapping those 50 posts' extractions into the graph: {w['entities_added']} entities added, {w['entities_removed']} removed, {w['relations_added']} relations added, {w['relations_removed']} removed"
)

re-extraction of 50 unchanged posts     entity J  median  relation J  median  type changes  calls   tokens
Qwen3_8-27B                                 0.92    1.00        0.74    0.85      7 of 675     108  168,037
Qwen3_8-Flash-Next                          0.61    0.62        0.19    0.13     45 of 500     108  145,460

entities per post: reference 13.94, drift model 12.94
swapping those 50 posts' extractions into the graph: 69 entities added, 107 removed, 344 relations added, 353 removed


`Qwen3_8-Flash-Next` keeps 61% of the reference model's entities and 19% of its relations (median 13%) on text that did not change, against 92% and 74% for the reference model run again. 45 of 500 shared entities also change type. Measured on these 50 posts with this prompt: a graph that mixes posts extracted by these two models disagrees with itself far more than either model disagrees with its own rerun. Re-extract a whole graph with one model, or keep the model name on every unit and compare like with like.

## Deduplication of near-duplicate names

The graph keys entities by lowercased name, so `access_log`, `access.log`, and `access logs` are three nodes. `fold_name` folds case, separators, dots inside a word, a trailing `()`, and a plural `s`. Names with any other punctuation are left alone, because on this corpus that punctuation is meaning: `rm -rf /*` and `rm -rf .*` must never merge, and neither should `-i` and `%i`. `https`, `windows`, and a few other words whose `s` is not a plural are excluded.

In [8]:
if RUN_EXPENSIVE:
    gm.write("dedup.json", gm.step_dedup(corpus))
    dedup = load("dedup.json")

print(
    f"{dedup['groups']:,} groups covering {dedup['entities_in_groups']:,} entities; {dedup['groups_with_type_disagreement']:,} groups mix entity types"
)
for group in dedup["largest_groups"][:4]:
    print("  ", " | ".join(group))
verdicts = {tuple(g["group"]): g for g in labels["groups"]}
counts = {"same": 0, "different": 0, "unclear": 0}
mentions = {"same": 0, "different": 0, "unclear": 0}
for row in dedup["label_sample"]:
    verdict = verdicts[tuple(row["group"])]["label"]
    counts[verdict] += 1
    mentions[verdict] += sum(row["mentions"])
print()
print(f"hand labels on {len(dedup['label_sample'])} groups drawn with seed 0: {counts}")
print(
    f"precision {counts['same'] / (counts['same'] + counts['different']):.3f}, by mentions {mentions['same'] / (mentions['same'] + mentions['different']):.3f}"
)
for g in labels["groups"]:
    if g["label"] == "different":
        print("   false merge:", " | ".join(g["group"]), "-", g["note"])

3,008 groups covering 6,588 entities; 1,044 groups mix entity types
   access_log | access.log | access logs | access log | accesslog | access_logs
   brute force attacks | brute force attack | brute-force attacks | bruteforce attacks | brute-force-attacks | brute-force attack
   cron job | cronjob | cron jobs | cronjobs | cron-job | cron.jobs
   log files | log file | logfile | log-files | logfiles | log-file

hand labels on 60 groups drawn with seed 0: {'same': 55, 'different': 3, 'unclear': 2}
precision 0.948, by mentions 0.700
   false merge: monitor | monitors - monitoring as a concept against monitor hardware
   false merge: proxy_pass | proxypass | proxy pass - nginx proxy_pass and Apache ProxyPass are different directives
   false merge: new | news - new and news are different words


The labels were made by reading each group's names, types, and mention counts, with no model call. The false merges are few, but one of them is large: nginx's `proxy_pass` and Apache's `ProxyPass` fold to the same key and carry 171 mentions between them, which is why precision weighted by mentions is lower. A rule this simple needs a review step for groups whose members disagree on type, which 1,044 groups do.

`merge_entities` keeps the name with the most mentions, unions every mention so provenance survives, takes the majority type, rewires relations to the survivor, combines relations that now share a key, and drops relations between two merged names.

In [9]:
m = dedup["merge"]
print(f"entities  {m['entities_before']:,} -> {m['entities_after']:,}")
print(
    f"relations {m['relations_before']:,} -> {m['relations_after']:,} ({m['relations_rewired']:,} rewired, {m['relations_collapsed']} combined, {m['self_loops_dropped']} loops dropped)"
)
d = m["topology_delta"]
print(
    f"isolated entities {d['isolated_entities']:+}, components {d['components']:+}, largest component share {d['largest_component_share']:+.4f}"
)
print(
    f"single-post entities {d['single_document_entities']:+,}; validation problems after the merge: {m['validation_problems_after']}"
)

entities  63,436 -> 59,856
relations 86,644 -> 86,121 (9,145 rewired, 491 combined, 32 loops dropped)
isolated entities -390, components -880, largest component share +0.0167
single-post entities -3,356; validation problems after the merge: 0


## A maintenance gate

`maintenance_gate` checks a changed graph before it replaces the one in service. Schema checks run `validate()` and require that no fact cites a retracted post. Topology checks compare entity loss, isolated entities, the largest component, and type shares with the graph before the change. Smoke questions ask the serving lookup questions about the retracted posts and fail if any answer cites one.

A smoke question is only useful if it cited the post before the change, so the gate fails a question that did not, as not armed. That rule caught a problem while this notebook was built. The package's `ExtractedGraphRetriever` linked an entity by any token it shared with the question, so on the 63,436-entity graph a question linked long error-message entities through words such as "with" and "a", and the retriever ranked the target post within its top 50 for one of the five questions. It now links an entity only when its whole name occurs in the question, skips names made only of stop words, and weights each entity by inverse document frequency. With that change it ranks the target post first for three of the five questions and third for the other two. The gate's smoke checks use `EntityLookup`, which links by whole name with the same weighting, because the stale-index scenario below needs a lookup built before the change and served after it.

In [10]:
if RUN_EXPENSIVE:
    gm.write("gate.json", gm.step_gate(corpus))
    gate = load("gate.json")

print("before the change, rank of the target post (EntityLookup, top 10; ExtractedGraphRetriever, top 50):")
for question, row in gate["smoke_before"].items():
    walker = gate["graph_retriever_rank_of_target_at_50"][question]
    print(f"  {row['target']:<8} {str(row['rank_of_target']):>4} {str(walker):>6}   {question}")
print()
scenarios = {
    "provenance_retraction": "retract() the five posts",
    "stale_serving_index": "retract(), lookup built before the change",
    "naive_delete": "delete every node and edge the posts touch",
    "mentions_only": "strip the posts' mentions, keep every fact",
    "dedup_merge": "merge all 3,008 duplicate groups",
}
for key, label in scenarios.items():
    result = gate[key]
    failed = [c for c in result["checks"] if not c["passed"]]
    print(f"{'PASS' if result['passed'] else 'FAIL'}  {label}")
    for check in failed[:2]:
        print(f"        {check['name'][:60]}: {check['detail'][:70]}")
    if len(failed) > 2:
        print(f"        and {len(failed) - 2} more failed checks")

before the change, rank of the target post (EntityLookup, top 10; ExtractedGraphRetriever, top 50):
  q9708       2      3   What is the difference between a pem file, a .der file, and .pkcs12?
  q62411      1      1   Pipe du -h into sort so the largest directories come first
  q309357     2      3   Use nmap to check a tcp port when ping only speaks icmp
  q264595     1      1   Can scp -r copy a sourcedirectory, or do I send a .tar file?
  q253313     1      1   ssh says bad owner or permissions on ~/.ssh/config; which chmod fixes it?

PASS  retract() the five posts
FAIL  retract(), lookup built before the change
        smoke: What is the difference between a pem file, a .der fil: cites 8 documents, retracted: q9708
        smoke: Pipe du -h into sort so the largest directories come : cites 9 documents, retracted: q62411
        and 3 more failed checks
FAIL  delete every node and edge the posts touch
        schema: validate(): 4116 problems, first: relation puttygen -configures->

The provenance retraction passes every check. The other three retraction strategies fail for different reasons, and each reason is one a team would hit:

- A serving index built before the change still cites all five posts, though the graph is correct.
- Deleting every node a post touches removes shared hubs such as `openssl` and `ssh`, leaves 4,116 problems including dangling relations, and shrinks the largest component from 77.9% to 76.1% of entities.
- Stripping mentions without dropping facts leaves 63 facts that cite nothing.

The merge passes because the gate is told how many names the merge is meant to fold away; without that it fails on entity loss, as a bound on unplanned loss should.

## LightRAG and HippoRAG 2

LightRAG 1.5.6 keeps the per-chunk extractions it made at insert time, so `adelete_by_doc_id` removes a document's chunks and rebuilds the entities and relations it shared with other documents from those cached extractions. `amerge_entities` merges nodes without a model call. An edit is a delete and an insert. All three ran live on a copy of the 5-passage MuSiQue index (`graphrag/musique/lightrag-5`), which was built with `Qwen3_6-35B-A3B`. The edit is this notebook's own change to one passage: it replaces one occurrence of "Maradona" with "Diego Armando Maradona".

In [11]:
if RUN_EXPENSIVE:
    gm.write("lightrag.json", gm.step_lightrag(RPM, model=LIGHTRAG_MODEL))
    lightrag = load("lightrag.json")

print(f"LightRAG {lightrag['lightrag_version']}, model {lightrag['model']}")
for step in lightrag["steps"]:
    print()
    print(step["operation"])
    print(f"  {step['model_calls']} model calls, {step['tokens']:,} tokens, {step['seconds']} s")
    print(f"  nodes {step['nodes'][0]} -> {step['nodes'][1]}, edges {step['edges'][0]} -> {step['edges'][1]}")
    if step["nodes_removed"]:
        print("  nodes removed:", ", ".join(step["nodes_removed"]))
    if step["nodes_added"]:
        print("  nodes added:  ", ", ".join(step["nodes_added"]))
    print(
        f"  descriptions rewritten: {len(step['descriptions_rewritten'])} ({', '.join(step['descriptions_rewritten'][:4])})"
    )
smoke = lightrag["smoke"]
print()
print(f"smoke: {smoke['question']}")
print(f"  cites {', '.join(smoke['cites'])}; cites the deleted passage: {smoke['cites_deleted']}")

LightRAG 1.5.6, model Qwen3_6-35B-A3B

adelete_by_doc_id(musique::00004)
  0 model calls, 0 tokens, 2.4 s
  nodes 61 -> 60, edges 65 -> 63
  nodes removed: Jesús Aranguren
  descriptions rewritten: 2 (Athletic Bilbao, Copa del Rey)

amerge_entities(['Barcelona', 'FC Barcelona'] -> FC Barcelona)
  0 model calls, 0 tokens, 11.0 s
  nodes 60 -> 59, edges 63 -> 61
  nodes removed: Barcelona
  descriptions rewritten: 1 (FC Barcelona)

edit musique::00002: adelete_by_doc_id then ainsert
  2 model calls, 6,291 tokens, 24.9 s
  nodes 59 -> 60, edges 61 -> 59
  nodes added:   Diego Armando Maradona
  descriptions rewritten: 10 (Bernd Schuster, Boca Juniors, Copa del Rey, Diego Maradona)

smoke: Which club did Jesús Aranguren spend his whole professional career with?
  cites musique::00000, musique::00001, musique::00002, musique::00003; cites the deleted passage: False


The delete rebuilt `Athletic Bilbao` and `Copa del Rey`, which the deleted passage shared with another, with no model call. The edit shows the hazard of re-extraction: the new text names "Diego Armando Maradona", the model extracted that as a new node, and the old `Diego Maradona` node stayed. One changed sentence created a near-duplicate, which the deduplication step above exists to catch.

atlas-rag, the HippoRAG 2 implementation RI #4 uses, has no operation that changes a built index. The cell lists what a search of its build and retrieval modules finds for delete, update, and merge, none of which edit an index, and the rebuild cost it leaves you with.

In [12]:
if RUN_EXPENSIVE:
    gm.write("hipporag.json", gm.step_hipporag())
    hipporag = load("hipporag.json")

print(f"atlas-rag {hipporag['atlas_rag_version']}: names matching {', '.join(hipporag['searched_for'])}")
for match in hipporag["matches"]:
    print("  ", match)
cost = hipporag["rebuild_cost"]
print(
    f"rebuild for any change: {cost['cost']['calls']:,} calls, {cost['cost']['total_tokens']:,} tokens, {cost['wall_seconds'] / 3600:.1f} hours on {cost['model']} for {cost['passages']:,} passages"
)

atlas-rag 0.0.5.post1: names matching delete, remove, update, upsert, incremental, merge, insert
   atlas_rag.kg_construction.triple_config:ProcessingConfig.remove_doc_spaces
   atlas_rag.kg_construction.triple_extraction:ProcessingConfig.remove_doc_spaces
   atlas_rag.kg_construction.triple_extraction:merge_csv_files
   atlas_rag.retriever.hipporag2:InferenceConfig.remove_unnecessary_rel
rebuild for any change: 11,285 calls, 7,422,640 tokens, 3.9 hours on Qwen3_6-35B-A3B for 11,656 passages


## Cost per change against a rebuild

In [13]:
lr = {s["operation"].split("(")[0].split(" ")[0]: s for s in lightrag["steps"]}
build = json.loads((Path.cwd() / "outputs" / "serverfault_graph_summary.json").read_text())
rows = [
    ("ServerFault graph", "retract one post", 0, 0),
    (
        "ServerFault graph",
        "revise one post",
        revision["edited_window_calls"] / revision["documents"],
        revision["edited_window_tokens"] / revision["documents"],
    ),
    ("ServerFault graph", "merge 3,008 name groups", 0, 0),
    ("ServerFault graph", "rebuild", build["cost"]["calls"], build["cost"]["total_tokens"]),
    (
        "LightRAG, 5 passages",
        "adelete_by_doc_id",
        lr["adelete_by_doc_id"]["model_calls"],
        lr["adelete_by_doc_id"]["tokens"],
    ),
    ("LightRAG, 5 passages", "amerge_entities", lr["amerge_entities"]["model_calls"], lr["amerge_entities"]["tokens"]),
    ("LightRAG, 5 passages", "edit (delete and insert)", lr["edit"]["model_calls"], lr["edit"]["tokens"]),
    (
        "HippoRAG 2, MuSiQue",
        "any change (rebuild)",
        hipporag["rebuild_cost"]["cost"]["calls"],
        hipporag["rebuild_cost"]["cost"]["total_tokens"],
    ),
]
print(f"{'index':<22} {'change':<26} {'calls':>8} {'tokens':>12}")
for index, change, calls, tokens in rows:
    print(f"{index:<22} {change:<26} {calls:>8,.0f} {tokens:>12,.0f}")

index                  change                        calls       tokens
ServerFault graph      retract one post                  0            0
ServerFault graph      revise one post                   1        1,321
ServerFault graph      merge 3,008 name groups           0            0
ServerFault graph      rebuild                      20,077   26,132,262
LightRAG, 5 passages   adelete_by_doc_id                 0            0
LightRAG, 5 passages   amerge_entities                   0            0
LightRAG, 5 passages   edit (delete and insert)          2        6,291
HippoRAG 2, MuSiQue    any change (rebuild)         11,285    7,422,640


The ServerFault rebuild figures are from `outputs/serverfault_graph_summary.json` (`Qwen3_8-27B`). A revision's calls depend on how many sections the edit touches. Each of the 25 measured revisions changed one section, because the sample takes one edited post per document.

## What these results cover

These results are one pass, one seed, on one graph. Each negative names what was tried.

- `ExtractedGraphRetriever` with any-token linking and one-hop expansion did not surface the target post in its top 50 on four of five questions on the full ServerFault graph. With whole-name linking and inverse-document-frequency weights, it ranks the target post third or better on all five. Five questions are a smoke test, and a retrieval comparison on this graph would need a question set with known target posts.
- The name-folding merge has false merges, and the largest is between two products. It would need a type or context check to separate `proxy_pass` from `ProxyPass`. The cheapest test is to merge only groups whose members share a type and relabel the same 60 groups.
- Relation changes after a revision are mostly not at the edit. Extraction that is stable on unchanged text would be needed for a relation diff to mean an edit. The cheapest test is to re-extract only the edited blocks and keep the unchanged blocks' relations, using the block ids every mention already carries.